In [2]:
import requests
import json

URL = "https://openlibrary.org"

SEARCH_ENDPOINT = "/search.json?q=test"
ENDPOINT = "/subjects/Fantasy.json"

headers = {
    "User-Agent": "open-library-api-data-pipeline (jakobgrob9@gmail.com)"
}

params = {
    "language": "en",
    "limit": 10
}

response = requests.get(
    URL + ENDPOINT,
    params=params, 
    headers=headers)

data = response.json()

with open('fantasyData.json', 'w') as f:
    json.dump(data, f, indent=4)

In [ ]:
fictionSubjects = ["Fantasy", "Historical Fiction", "Horror", "Humor", "Literature", 
            "Magic", "Mystery and detective stories", "Plays", "Poetry", "Romance", 
            "Science Fiction", "Short Stories", "Thriller", "Young Adult"]

### Loading data from .json file

In [2]:

import json
with open('dumps/fantasyData.json', 'r') as f:
    data = json.load(f)

### Transforming json response

In [ ]:
import pandas as pd

def get_work(data: dict) -> list:
    
    work = []

    name = data.get('name')
    works = data.get('works', [])
    for item in works:
        work_key = item.get('key').split("/")[-1]
        title = item.get('title')
        cover_id = item.get('cover_id')
        cover_edition_key = item.get('cover_edition_key')
        first_publish_year = item.get('first_publish_year') 
        work.append({
            "WorkKey": work_key,
            "Title": title,
            "CoverId": cover_id,
            "CoverEditionKey": cover_edition_key,
            "Subject": name,
            "FirstPublishYear": first_publish_year
        })
    return work

work = get_work(data)
work[0]

# work_df = pd.DataFrame(get_work(data))
# work_df.head()

#### Don't forget to create mapping table for
* subject
* authors
#### keys in from the 'works' key from the response

### Postgres connection

In [ ]:
import psycopg2
    
conn = psycopg2.connect(
    host="localhost",
    database="open-library",
    user="postgres",
    password="postgres"
)
cursor = conn.cursor()
# cursor.execute("INSERT INTO test (num, data) VALUES (%s, %s)", (100, "abc'def"))

cursor.execute("SELECT * FROM work;")
one = cursor.fetchone()

display(one)
# Close communication with the database
cursor.close()
conn.close()

In [5]:
work_df

,WorkKey,Title,CoverId,CoverEditionKey,Subject,FirstPublishYear
0,OL138052W,Alice's Adventures in Wonderland,10527843,OL31754751M,fantasy,1865
1,OL18417W,The Wonderful Wizard of Oz,552443,OL8155451M,fantasy,1899
2,OL24034W,Treasure Island,13859660,OL40812029M,fantasy,1880
3,OL20600W,Gulliver's Travels,12717083,OL26445784M,fantasy,1726
4,OL259010W,A Midsummer Night's Dream,7205924,OL24594641M,fantasy,1600
5,OL1089297W,The Prince,12726168,OL37826838M,fantasy,1515
6,OL151406W,Through the Looking-Glass,11272464,OL21298716M,fantasy,1865
7,OL28570037W,The Wind in the Willows,13335427,OL40046991M,fantasy,1908
8,OL99499W,Five Children and It,28174,OL9232844M,fantasy,1905
9,OL15449W,The Princess and the Goblin,14363454,OL48579872M,fantasy,1872


### Ofc we will do that with database.ini or different way but for now it's ok
### inserting multiple rows

In [ ]:
def insert_many_vendors(vendor_list):
    """ Insert multiple vendors into the vendors table  """

    sql = "INSERT INTO vendors(vendor_name) VALUES(%s) RETURNING *"
    config = load_config()
    try:
        with  psycopg2.connect(**config) as conn:
            with  conn.cursor() as cur:
                # execute the INSERT statement
                cur.executemany(sql, vendor_list)

            # commit the changes to the database
            conn.commit()
    except (Exception, psycopg2.DatabaseError) as error:
        print(error)